# Beyond RAG — VINE (Colab)

Every code cell is numbered at the top: `# CELL n`. Numbers count CODE cells
only, top to bottom, and are stable.

**After any restart:** 1, 2, 3, 4, then 9, 10, 11 (model), 12 (wire VINE).
Corpus cells 5, 6, 7 are done — skip them.

**T4 is enough** for cells 1-8 and 13. The only heavy local model is the
ColQwen image embedder; turn it off with
`init_pipeline(load_image_embedder=False)` and a T4 is fine. The VLM is an API
call, so it loads nothing locally.

    Kq = Retrieve(q, K)                  Eq 1
    Nq = Compile(q, Kq)                  Eq 1, S3.1
    validate(Nq)                         before anything executes
    execute under certificate gating     Eq 4
    a = f_LM(q, C*, Pi_q)                Eq 6, only after certification


## 0. Setup — every session

In [ ]:
# CELL 1
# COLAB ONLY — skip this cell if running on HPRC.
import sys, os

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

# ── WHERE EVERYTHING LIVES ─────────────────────────────────────────────────
# Data (PDF, cache, figures, page images, HF cache, snapshots, runs) lives on
# Drive. The repo is cloned to local disk and is disposable.
#
# MRAG_BASE_DIR overrides the default in config.py. It must be set BEFORE any
# mrag import, and it must be an env var so subprocesses (ingest_v4.py, run via
# !python) inherit it — sys.modules detection does not cross process boundaries.
DRIVE_DIR = "/content/drive/MyDrive/Beyond_RAG"       # <- data on Drive
REPO_DIR  = "/content/Beyond_RAG_repo"                # <- code, local, disposable
REPO_URL  = "https://github.com/hannanazad/Beyond_RAG.git"

os.environ["MRAG_ENV"]      = "colab"
os.environ["MRAG_BASE_DIR"] = DRIVE_DIR
os.makedirs(DRIVE_DIR, exist_ok=True)

if not os.path.isdir(REPO_DIR):
    !git clone $REPO_URL $REPO_DIR
else:
    !cd $REPO_DIR && git pull

sys.path.insert(0, REPO_DIR)
print(f"\ndata : {DRIVE_DIR}")
print(f"code : {REPO_DIR}")

# Step 0 — poppler-utils.
# Colab ships a PARTIAL poppler install: pdftotext and pdftoppm are present
# but pdftohtml is NOT. mrag/font_index.py needs pdftohtml to read the font
# a caption is set in; without it, caption anchors cannot be validated and
# body-text mentions get cropped as figures.
!apt-get install -y -qq poppler-utils
!which pdftotext pdftoppm pdftohtml pdffonts

# Step 1 — torch matched to Colab's CUDA 12.4.
# Colab ships torch 2.5.1, but transformers >=4.51 enforces CVE-2025-32434 and
# refuses to load .bin checkpoints on torch <2.6. BGE-M3 ships .bin.
!pip install -q --index-url https://download.pytorch.org/whl/cu124 \
    torch==2.6.0 torchvision==0.21.0

# Step 2 — everything else
!pip install -q -r $REPO_DIR/requirements.txt

# Step 3 — pip's bulk resolver does not enforce CEILINGS when an installed
# version already satisfies the lower bound. Colab ships newer packages than
# this pipeline can use, so pin them explicitly.
!pip install -q --no-deps --force-reinstall \
    "transformers>=4.49,<4.55" \
    "huggingface_hub>=0.34,<0.35" \
    "tokenizers>=0.21,<0.22" \
    "torchao>=0.13,<0.14"

!python -c "from transformers import PreTrainedModel; print('transformers import OK')"

In [ ]:
# CELL 2
# API keys from Colab Secrets (key icon, left sidebar).
# Keys authenticate only; select the model with CFG.set_vlm_model(...).
from google.colab import userdata
import os

def _load(env_name, *secret_names):
    for s in secret_names:
        try:
            os.environ[env_name] = userdata.get(s)
            print(f"{env_name}: loaded (secret {s!r})")
            return
        except Exception:
            continue
    print(f"{env_name}: no matching secret (skipped)")

_load("DASHSCOPE_API_KEY", "DASHSCOPE_API_KEY", "QWEN")
_load("ANTHROPIC_API_KEY", "ANTHROPIC_API_KEY")
#_load("GEMINI_API_KEY",    "GEMINI_API_KEY")

In [ ]:
# CELL 3
import os
from pathlib import Path

old = Path("/content/drive/MyDrive/MRAG")
new = Path(os.environ["MRAG_BASE_DIR"])

def survey(p):
    if not p.exists():
        return "does not exist"
    bits = []
    for name in ("mmrag_cache_v3", "figures", "page_images", "hf_cache"):
        d = p / name
        bits.append(f"{name}={'yes' if d.exists() else 'no'}")
    pdfs = list(p.glob("*.pdf"))
    bits.append(f"pdf={pdfs[0].name if pdfs else 'MISSING'}")
    bits.append(f"snapshot={'yes' if (p/'qdrant_db.tar').exists() else 'no'}")
    return "  ".join(bits)

print(f"OLD  {old}\n     {survey(old)}\n")
print(f"NEW  {new}\n     {survey(new)}\n")

if old.exists() and not (new / "mmrag_cache_v3").exists():
    print("ACTION: rename MRAG -> Beyond_RAG in the Drive web UI, then re-run cell 0.")
    print("        Copying through Colab works but is slow — page_images alone is ~3 GB.")
elif (new / "mmrag_cache_v3").exists():
    print("Beyond_RAG already holds the cache. Nothing to do.")
else:
    print("Neither folder has a cache. Upload the MUTCD PDF to Beyond_RAG and ingest.")

In [ ]:
# CELL 4
import os, sys
REPO_DIR = "/content/Beyond_RAG_repo"
sys.path.insert(0, REPO_DIR if os.path.isdir(REPO_DIR) else ".")

from mrag.config import CFG

print("Environment :", CFG.environment)
print("Base dir    :", CFG.base_dir, "| exists:", CFG.base_dir.exists())
print("PDF path    :", CFG.pdf_path, "| exists:", CFG.pdf_path.exists())
print("Cache dir   :", CFG.cache_dir)
print("Qdrant dir  :", CFG.qdrant_dir, "  (local SSD, snapshotted to Drive)")
print("HF cache    :", CFG.hf_home)
print("VLM provider:", CFG.vlm_provider)
print("VLM model   :", CFG.vlm_model_api if CFG.vlm_provider == "api" else CFG.vlm_model)
print("API key set :", bool(os.environ.get(CFG.api_key_env_var)))
print()
print("Image budget: max_sheets_per_figure =", CFG.max_sheets_per_figure,
      "| max_images_total =", CFG.max_images_total,
      "| max_page_images =", CFG.max_page_images)

assert str(CFG.base_dir).endswith("Beyond_RAG"), \
    f"base_dir is {CFG.base_dir} — MRAG_BASE_DIR did not take. Re-run cell 0."

try:
    import torch
    print("GPU         :", torch.cuda.get_device_name(0))
except Exception as e:
    print("GPU         : none —", e)

assert CFG.pdf_path.exists(), (
    f"No PDF at {CFG.pdf_path} or anywhere in {CFG.base_dir}. "
    f"Upload the MUTCD PDF there first."
)

## 1. Corpus — DONE, skip

Cells 5, 6 and 7 were run on 2026-09-19. Run them again only if the corpus
changes.

In [ ]:
# CELL 5
# The 22 chunks are footnotes the extractor never read, taken from the
# verified transcription instead. Without them, six tables -- 2C-6 and
# 4C-1..4C-5 -- have footnotes that no chunk anywhere contains, so a
# certificate citing one has nothing to point at and no search can find it.
# 4C-1 note c and the 4C-4/4C-5 scope notes are in that group.
from pathlib import Path
from mrag.config import CFG
import json

NEW = CFG.cache_dir / "table_notes_new.jsonl"      # put the file here first
CH  = CFG.chunks_jsonl

assert NEW.exists(), f"{NEW} not found -- upload it to the cache directory"
have = {json.loads(l)["chunk_id"] for l in CH.open()}
add  = [json.loads(l) for l in NEW.open()]
todo = [c for c in add if c["chunk_id"] not in have]

print(f"chunks now      : {len(have)}")
print(f"in the patch    : {len(add)}")
print(f"not yet present : {len(todo)}")

if todo:
    with CH.open("a") as fh:
        for c in todo:
            fh.write(json.dumps(c) + "\n")
    print(f"appended        -> {sum(1 for _ in CH.open())} lines")
else:
    print("nothing to do; already applied")


In [ ]:
# CELL 6
# mutcd_tables.jsonl is DATA, not code -- it is not in the repo. Find it.
import sys
sys.path.insert(0, "/content/Beyond_RAG_repo")
from pathlib import Path
from mrag.config import CFG

CANDIDATES = [
    CFG.cache_dir / "mutcd_tables.jsonl",          # alongside chunks.jsonl
    CFG.base_dir  / "mutcd_tables.jsonl",
    Path("/content/Beyond_RAG_repo/mutcd_tables.jsonl"),
    Path("/content/drive/MyDrive/Beyond_RAG/mutcd_tables.jsonl"),
]
TABLES = next((p for p in CANDIDATES if p.exists()), None)

if TABLES is None:
    print("NOT FOUND. Looked in:")
    for p in CANDIDATES:
        print("  ", p)
    print("\nSearching Drive (may take a moment):")
    hits = list(Path("/content/drive/MyDrive").rglob("mutcd_tables.jsonl"))
    for h in hits:
        print("  found:", h)
    if hits:
        TABLES = hits[0]
        print("\nusing:", TABLES)

assert TABLES is not None, "upload mutcd_tables.jsonl to CFG.cache_dir"
print("TABLES =", TABLES)

# Every one of the 230 footnotes should carry a chunk_id, so a calculator
# certificate can cite the specific note that governs a value rather than the
# table's notes as a blob.
from mrag.vine.table_data import load as load_tables
ts = load_tables(TABLES)
missing = [(t.table_id, f.marker) for t in ts for f in t.footnotes if not f.chunk_id]
print(f"records {len(ts)} | footnotes {sum(len(t.footnotes) for t in ts)}"
      f" | without a chunk_id {len(missing)}")
print("LINKED version" if not missing
      else f"OLD version -- replace it. First few: {missing[:5]}")

In [ ]:
# CELL 7
!cd /content/Beyond_RAG_repo && python scripts/ingest_v4.py

In [ ]:
# CELL 8
import json, collections
from mrag.config import CFG

rows = [json.loads(l) for l in CFG.chunks_jsonl.open()]
print("chunks      :", len(rows))
print("by source   :", dict(collections.Counter(r.get("source") for r in rows)))
print("minted      :", sum(1 for r in rows if r.get("origin") == "verified_transcription"))
print("authority_inferred present:",
      sum(1 for r in rows if "authority_inferred" in r), "/", len(rows))

import pickle
g = pickle.load(CFG.graph_pickle.open("rb"))
print("graph       :", g.number_of_nodes(), "nodes,", g.number_of_edges(), "edges")

# the paper's 4.1 sentence needs these six numbers
secs = sum(1 for n in g.nodes if n.startswith("section:"))
figs = sum(1 for n in g.nodes if n.startswith("figure:") and "Table" not in n)
tabs = sum(1 for n in g.nodes if n.startswith("figure:Table"))
print(f"\nfor paper 4.1: {secs} sections, {len(rows)} chunks, {figs} figures, "
      f"{tabs} tables, {g.number_of_nodes()} nodes, {g.number_of_edges()} edges")


## 2. Initialise

`init_pipeline()` is a process-wide singleton AND Qdrant takes an exclusive
file lock, so **the first call in a kernel decides everything** and a second
one cannot be made. Cell 9 is therefore the only place the pipeline is built.

Pass `load_image_embedder=False` to fit on a T4. Pass `load_vlm=False` only if
you will not call a model this session.

In [ ]:
# CELL 9
import logging, os
logging.basicConfig(level=logging.INFO, format="%(levelname)-7s %(name)s :: %(message)s")

from mrag.ask import init_pipeline
from mrag.config import CFG

T4_MODE = False          # True -> skip ColQwen, ~4 GB, fits a T4
NEED_MODEL = True        # False -> retrieval only, no API calls at all

pipeline = init_pipeline(load_image_embedder=not T4_MODE, load_vlm=NEED_MODEL)

print("store   :", CFG.qdrant_dir)
print("graph   :", pipeline.kg.g.number_of_nodes(), "nodes,",
                   pipeline.kg.g.number_of_edges(), "edges")
print("image   :", "loaded" if pipeline.image else "OFF (T4 mode)")
print("vlm     :", pipeline.vlm.loaded_name if pipeline.vlm else "not loaded")


### 2.1 Choose the model

In [ ]:
# CELL 10
# ── List every VLM the config knows about, grouped by provider ──────────────
import os
from mrag.config import (
    CFG, VLM_API_MODELS, VLM_PROVIDERS, provider_of_model,
    VLM_TEXT_ONLY_ALIASES, VLM_VISION_UNVERIFIED,
)

# key present?  ->  can you actually call it
have_key = {p: bool(os.environ.get(v["env_var"])) for p, v in VLM_PROVIDERS.items()}
print("API keys loaded:", {p: ("yes" if k else "NO") for p, k in have_key.items()})
print("Currently selected:", CFG.vlm_model_api, f"[{provider_of_model(CFG.vlm_model_api)}]")
print()

# one row per distinct model id, collecting the aliases that point at it
by_model = {}
for alias, mid in VLM_API_MODELS.items():
    by_model.setdefault(mid, []).append(alias)

for prov in ("anthropic", "gemini", "dashscope"):
    rows = [(m, a) for m, a in by_model.items() if provider_of_model(m) == prov]
    if not rows:
        continue
    print(f"── {prov.upper()}   (key: {'yes' if have_key[prov] else 'NO'})")
    for mid, aliases in sorted(rows):
        flags = []
        if mid in VLM_VISION_UNVERIFIED:
            flags.append("VISION UNVERIFIED")
        if any(a in VLM_TEXT_ONLY_ALIASES for a in aliases):
            flags.append("TEXT ONLY - will 400 on images")
        if mid == CFG.vlm_model_api:
            flags.append("<< SELECTED")
        note = ("   " + " | ".join(flags)) if flags else ""
        print(f"   {'  '.join(sorted(aliases)):<42} -> {mid}{note}")
    print()

print("Select with:  CFG.set_vlm_model('fast_claude')")
print("A raw model id works too:  CFG.set_vlm_model('claude-sonnet-5')")

In [ ]:
# CELL 11
# ── Pick a model ────────────────────────────────────────────────────────────
resolved = CFG.set_vlm_model("frontier_claude")      # cheap, for smoke tests

print("model    :", resolved)
#print("provider :", provider_of_model(resolved))
print("key set  :", bool(os.environ.get(CFG.api_key_env_var)), f"({CFG.api_key_env_var})")

# The pipeline reads CFG at call time, so no re-init needed.
#_ = ask("What shape and colour is a STOP sign?")

### 2.2 Wire the model into VINE

In [ ]:
# CELL 12
from pathlib import Path
from mrag.vine import make_ask, build_verifiers
from mrag.vine.table_data import load as load_tables
from mrag.config import CFG

CANDIDATES = [CFG.cache_dir / "mutcd_tables.jsonl",
              CFG.base_dir  / "mutcd_tables.jsonl",
              Path("/content/Beyond_RAG_repo/mutcd_tables.jsonl")]
TABLES = next((p for p in CANDIDATES if p.exists()), None)
if TABLES is None:
    hits = list(Path("/content/drive/MyDrive").rglob("mutcd_tables.jsonl"))
    TABLES = hits[0] if hits else None
assert TABLES is not None, "mutcd_tables.jsonl not found"

tables = load_tables(TABLES)
missing = [(t.table_id, f.marker) for t in tables for f in t.footnotes if not f.chunk_id]
ask = make_ask(pipeline.vlm, max_tokens=1200) if pipeline.vlm else None

print("tables    :", len(tables), "records from", TABLES)
print("linked    :", "yes" if not missing else f"NO -- {missing[:3]}")
print("model     :", pipeline.vlm.loaded_name if pipeline.vlm else "none")
print("verifiers :", sorted(build_verifiers(tables=tables, kg=pipeline.kg,
                                            retriever=pipeline.retriever, ask=ask)))


## 3. Did retrieval supply what the answer needs?  — FREE, no model

**Run this before any dry run.** It answers the question the dry run cannot:
when an obligation is missing from a network, was it dropped by the parser, or
was it never retrieved in the first place?

Set `QUESTION` and `NEEDED` together. `NEEDED` is the list of provisions YOU
know the correct answer rests on, read out of the manual. Without it there is
nothing to check against.

In [ ]:
# CELL 13
QUESTION = "Which horizontal alignment sign is required in advance of this curve?"

# The provisions the correct answer needs. Worked out by hand from the manual.
#   2C.06 P1  Chart A decides IF devices are needed; Chart B decides the type
#   2C.06 P4  may be omitted when approach speed limit <= 20 mph
#   2C.06 P5  may be omitted on urban streets with AADT <= 1,000
#   2C.07 P1  the sign SHALL be a Curve (W1-2) unless this Section says otherwise
#   2C.07 P2  Turn (W1-1) if advisory speed <= 30 mph
#   2C.07 P3  Reverse Turn / Reverse Curve if two opposite changes < 600 ft apart
#   2C.07 P5  Winding Road (W1-5) if three or more changes < 600 ft apart
#   2C.07 P7  Hairpin Curve (W1-11) if the change is >= 135 degrees
#   2C.07 P8  270-degree Loop (W1-15) if the change is about 270 degrees
NEEDED = {"2C.06": [1, 4, 5], "2C.07": [1, 2, 3, 5, 7, 8]}

got = pipeline.retriever.retrieve_for_compile(QUESTION)
have = {(c["section_id"], int(c["ordinal"])) for c in got.chunks}

print(f"{len(got.chunks)} chunks, {len(got.figures)} figures\n")
miss = 0
for sec, paras in NEEDED.items():
    for p in paras:
        ok = (sec, p) in have
        miss += not ok
        print(f"  {sec} para {p}: {'present' if ok else 'MISSING'}")
print(f"\n{len(NEEDED and [p for v in NEEDED.values() for p in v]) - miss}"
      f"/{sum(len(v) for v in NEEDED.values())} needed provisions retrieved")
if miss:
    print("\n-> the parser CANNOT include what it never saw. Fix retrieval first.")

print("\n--- everything that came back ---")
for c in got.chunks:
    print(f"[{c['section_id']} {c['content_type']} #{c['ordinal']}] {c.get('text','')[:130]}")


## 4. Dry run — compile a network

One model call (the parser). No verifiers, no answer call.

Only meaningful once cell 13 shows the needed provisions were retrieved.

In [ ]:
# CELL 14
from mrag.vine import ask_vine

r = ask_vine(QUESTION,
             retriever=pipeline.retriever,
             tables=tables, kg=pipeline.kg, ask=ask,
             dry_run=True)
print(r.summary())

# read these three before anything else:
#   compiled by : must be semantic_parser. "baseline" = the model failed.
#   obligations : more than ~5 for a real section.
#   verifiers   : a spread. all-llm means the split was not atomic.


### 4.1 The obligations

In [ ]:
# CELL 15
for o in r.spec.obligations:
    op = r.network.op(o.id)
    guard = f"  guard: {o.guard.describe()}" if o.guard else ""
    print(f"{o.id:4} [{o.authority:9}] {op.verifier:<26} {o.type:<16}"
          f" requires={o.requires}{guard}")
    print(f"     {o.claim[:110]}")
    if o.evidence_hint:
        print(f"     hints: {o.evidence_hint}")
print()
for m in r.spec.merges:
    print(f"{m.id:4} {m.kind:<12} inputs={m.inputs}")
print(f"\nterminal: {r.spec.terminal}")

# hints must read "Table 2C-4", not "2C-4". Bare ids mean stale code: restart.


### 4.2 Structural checks the validator cannot make

A network can be perfectly valid and still be wrong. These three are the ones
that have actually gone wrong, so they are checked here rather than by eye.

In [ ]:
# CELL 16
from mrag.vine.answer import supporting_certificates
from mrag.vine import execute, Certificate, Status

stub = lambda op, st: Certificate(claim=op.claim, status=Status.TRUE, confidence=1.0,
                                  normative_authority=op.normative_authority)
t = execute(r.network, {k: stub for k in
            ("llm", "vlm", "calculator", "symbolic", "cross_reference_resolver")})
reaches = {c.obligation_id for c in supporting_certificates(r.network, t)}
dangling = sorted({o.id for o in r.network.operations} - reaches)

print("waves     :", t.waves)
print("depth     :", t.synchronization_depth, "for", t.operations_run, "operations")
print("DANGLING  :", dangling or "none")
if dangling:
    print("  -> these are verified and then DISCARDED; they cannot affect the answer")
    for oid in dangling:
        op = r.network.op(oid)
        print(f"     {oid}: {op.claim[:90]}")

print("\nexception merges (inputs[0] is the BASE RULE, the rest RELAX it):")
for m in r.spec.merges:
    if m.kind == "exception":
        base, *relax = m.inputs
        b = r.network.op(base)
        print(f"  {m.id}: base = {base} — {(b.claim if b else '(merge)')[:70]}")
        for x in relax:
            o = r.network.op(x)
            print(f"        relaxed by {x} — {(o.claim if o else '(merge)')[:66]}")
        print("     ask: does each of those mean the base need NOT be satisfied?")
        print("          a provision that changes HOW it is met is not an exception.")


### 4.3 The parser report

In [ ]:
# CELL 17
from collections import Counter
print("verifier split:", dict(Counter(o.verifier for o in r.network.operations
                                      if o.merge is None)))
print("parse attempts:", r.report.attempts, "| source:", r.report.source)
for group in r.report.problems:
    print("  rejected:", group)
for note in r.report.notes:
    print("  note    :", note)

if r.report.raw:
    print("\n--- raw parser reply ---")
    print(r.report.raw[-1][:1500])
else:
    print("\n(no model reply at all -- see the problems above. A missing API key")
    print(" shows up here as 'the model call failed' plus source: baseline.)")


### 4.4 Did it find the provisions you listed?

Scores the network against the same `NEEDED` list cell 13 used, by matching on
`source_chunk`. This is the number that says whether the decomposition is
right, rather than merely valid.

In [ ]:
# CELL 18
import json
from mrag.config import CFG

rows = {c["chunk_id"]: c for c in map(json.loads, CFG.chunks_jsonl.open())}
covered = set()
for o in r.spec.obligations:
    c = rows.get(o.source_chunk)
    if c:
        covered.add((c["section_id"], int(c["ordinal"])))

total = sum(len(v) for v in NEEDED.values())
hit = 0
for sec, paras in NEEDED.items():
    for p in paras:
        ok = (sec, p) in covered
        hit += ok
        print(f"  {sec} para {p}: {'covered' if ok else 'NOT COVERED'}")
print(f"\n{hit}/{total} needed provisions became obligations")
extra = sorted(covered - {(s, p) for s, v in NEEDED.items() for p in v})
if extra:
    print("obligations from provisions not on your list:", extra)


## 5. Live run — verifiers execute

Roughly one model call per obligation, plus the answer call. Do not run this
until cells 13, 16 and 18 look right.

In [ ]:
# CELL 19
r = ask_vine(QUESTION,
             retriever=pipeline.retriever,
             tables=tables, kg=pipeline.kg, ask=ask,
             record_dir=str(CFG.base_dir / "vine_runs"))
print(r.summary())
print("\n" + "=" * 70)
print(r.answer.text)
print("=" * 70)
print("\ncitations:")
for c in r.answer.citations:
    print("  ", c)
if r.answer.unresolved:
    print("\nleft unresolved:")
    for u in r.answer.unresolved:
        print("  ", u)


### 5.1 Every certificate behind the decision

In [ ]:
# CELL 20
from mrag.vine import supporting_certificates
for c in supporting_certificates(r.network, r.trace):
    print(f"[{c.status.value:7}] {c.normative_authority.value:9} "
          f"{c.verifier:<26} conf {c.confidence:.2f}")
    print(f"   {c.claim[:95]}")
    print(f"   evidence: {[(e.type, e.id) for e in c.evidence]}")
    reason = (c.provenance or {}).get("reason")
    if reason:
        print(f"   reason  : {reason}")
    print()
print("unsupported certification:", r.trace.unsupported_certification())


### 5.2 Failures that do not crash

In [ ]:
# CELL 21
for c in r.trace.store.all():
    p = c.provenance or {}
    flags = [k for k in ("fabricated_evidence_ids", "downgraded",
                         "partial_figures", "undischarged") if k in p]
    if flags:
        print(f"{c.obligation_id:6} {c.status.value:8} {flags}")
        for f in flags:
            print(f"    {f}: {p[f]}")


## 6. Batch

In [ ]:
# CELL 22
QUESTIONS = [
    # put your questions here
    "Which horizontal alignment sign is required in advance of this curve?",
]

import json
from pathlib import Path
OUT = Path(CFG.base_dir) / "vine_runs"
results = []
for q in QUESTIONS:
    res = ask_vine(q, retriever=pipeline.retriever, tables=tables,
                   kg=pipeline.kg, ask=ask, record_dir=str(OUT))
    results.append(res)
    print(res.summary()); print("-" * 70)

from collections import Counter
print("\nstage     :", dict(Counter(x.stage for x in results)))
print("decision  :", dict(Counter(x.status.value for x in results)))
print("certified :", sum(x.certified for x in results), "/", len(results))
print("compiled  :", dict(Counter(x.report.source for x in results if x.report)))
print("UCR       :", sum(1 for x in results if x.trace
                         and x.trace.unsupported_certification()), "/", len(results))


In [ ]:
# CELL 23
strict = [ask_vine(q, retriever=pipeline.retriever, tables=tables,
                   kg=pipeline.kg, ask=ask, dry_run=True,
                   fall_back_to_baseline=False) for q in QUESTIONS]
ok = [x for x in strict if x.report and x.report.source == "semantic_parser"]
print(f"parser produced a valid network unaided: {len(ok)} / {len(strict)}")
print("attempts needed:", [x.report.attempts for x in ok])


## 7. The measured tables — FREE, no model

Simulated verifier outcomes through the real executor. `terminal_accuracy` is
agreement with the network's own logic — NOT the paper's Full credit.

In [ ]:
# CELL 24
import json, random
from mrag.vine.compile import compile_section, instantiate, CompileError
from mrag.vine.experiments import table3, table5, format_table
from mrag.vine.faults import table4

chunks = [json.loads(l) for l in CFG.chunks_jsonl.open()]
secs = sorted({c["section_id"] for c in chunks})
random.Random(7).shuffle(secs)

nets = []
for s in secs:
    try:
        spec = compile_section(s, chunks)
    except CompileError:
        continue
    n, p = instantiate(spec)
    if not p and len(n.operations) >= 4:
        nets.append(n)
    if len(nets) >= 200:
        break
print("networks:", len(nets))

print("\n=== TABLE 3 — linear vs dependency-aware ===")
print(format_table(table3(nets, range(15)),
                   ["terminal_accuracy", "UCR", "calls", "latency"]))
print("\n=== TABLE 5 — ablation ===")
print(format_table(table5(nets, range(15)), ["terminal_accuracy", "UCR", "PC"]))
print("\n=== TABLE 4 — fault containment ===")
print(format_table(table4(nets[:80], range(8)),
                   ["text", "numeric", "visual", "answer preserved"]))


In [ ]:
# CELL 25
# Collect networks from the SEMANTIC parser rather than the printed structure.
parsed = [x.network for x in strict
          if x.network and x.report and x.report.source == "semantic_parser"]
print("parsed networks:", len(parsed),
      "| carrying guards:", sum(1 for n in parsed
                                for o in n.operations if o.guard))
if parsed:
    print(format_table(table5(parsed, range(15)),
                       ["terminal_accuracy", "UCR", "PC"]))


## 8. Diagnostics

### 8.1 Per-obligation retrieval (Eq 5)

In [ ]:
# CELL 26
r = pipeline.retriever
print("new methods:",
      hasattr(r, "retrieve_for_obligation"),
      hasattr(pipeline.store, "fetch_chunks_by_ids"),
      hasattr(pipeline.kg, "chunks_for_section"))

print("4K.04 chunks:", len(pipeline.kg.chunks_for_section("4K.04")))
print("4K.04 cites :", pipeline.kg.sections_cited_by("4K.04"))

res = r.retrieve_for_obligation(
    query="Does this push button installation comply with 4K.04?",
    obligation="the locator tone repeats at 1-second intervals",
    certificates=[{"evidence": [{"type": "section", "id": "4K.04"}]}],
)
print("\nanchors:", res.debug["anchor_sections"], "| anchor chunks:", res.debug["anchor_chunks"])
for c in res.chunks:
    print(f"  {c['section_id']:<9} {c['content_type']:<9} {c.get('text','')[:60]}")

### 8.2 Compile-time retrieval

In [ ]:
# CELL 27
r = pipeline.retriever
q = "minimum sizes for regulatory signs on multi-lane conventional roads"
on  = r.retrieve_for_compile(q)
off = r.retrieve_for_compile(q, expand_cross_references=False)
print("expanded kept:", on.debug["n_expanded_kept"], "of", on.debug["n_chunks"])
print("with    :", sorted({c['section_id'] for c in on.chunks}))
print("without :", sorted({c['section_id'] for c in off.chunks}))
print("gained  :", sorted({c['section_id'] for c in on.chunks} - {c['section_id'] for c in off.chunks}))

### 8.3 Table and figure notes

In [ ]:
# CELL 28
import json, collections
rows = [json.loads(l) for l in open(CFG.chunks_jsonl) if l.strip()]
notes = [r for r in rows if r.get("source") in ("table_note", "figure_note")
         and r.get("authority_inferred")]
print(f"notes printed inside crops: {len(notes)}")
print("  from tables :", len({r['parent_id'] for r in notes if r['source']=='table_note'}), "tables")
print("  from figures:", len({r['parent_id'] for r in notes if r['source']=='figure_note'}), "figures")
print("  inferred type:", dict(collections.Counter(r["content_type"] for r in notes)))

print("\nnotes that carry an obligation (shall / should):")
for r in [x for x in notes if x["content_type"] in ("Standard", "Guidance")][:8]:
    print(f"   [{r['content_type']:<8}] {r['text'][:115]}")

### 8.4 What did retrieval return?

In [ ]:
# CELL 29
from mrag.retrieval import Retriever

res = pipeline.retriever.retrieve("STOP sign sizes at an all-way stop")

print(f"{len(res.chunks)} chunks\n")
for c in res.chunks:
    print(f"  {c.get('section_id'):<10} {c.get('content_type'):<9} "
          f"p.{c.get('page_printed'):<5} score={c.get('score', 0):.3f}")

print(f"\n{len(res.figures)} figures")
for f in res.figures:
    n = len(f.get("image_paths") or [])
    print(f"  {f.get('figure_id'):<16} sheets={n:<3} source={f.get('source','?')}")

### 8.5 Knowledge graph shape

In [ ]:
# CELL 30
import collections
kg = pipeline.kg
print(kg.g.number_of_nodes(), "nodes,", kg.g.number_of_edges(), "edges")
print()
print("node kinds:")
for k, v in collections.Counter(d.get("kind", "?") for _, d in kg.g.nodes(data=True)).most_common():
    print(f"  {k:<14} {v}")
print()
print("edge labels:")
for k, v in collections.Counter(d.get("label", "?") for *_, d in kg.g.edges(data=True)).most_common():
    print(f"  {k:<20} {v}")

### 8.6 Cross-references from a section

In [ ]:
# CELL 31
# Sections cross-referenced by a given section (cites_section edges).
target = "2B.04"
refs = set()
for c in (json.loads(l) for l in open(CFG.chunks_jsonl) if l.strip()):
    if c["section_id"] == target:
        refs.update(c.get("section_refs") or [])
print(f"{target} cross-references: {sorted(refs)}")

### 8.7 Is the sparse leg alive?

In [ ]:
# CELL 32
import json
s = json.load(open(CFG.cache_dir / "chunks_sparse.json"))
print(len(s), "entries |", sum(1 for x in s if x), "non-empty")

### 8.8 Inspect a table's crops

In [ ]:
# CELL 33
from IPython.display import display, Image as IPImage
import json
figs = [json.loads(l) for l in open(CFG.figures_jsonl) if l.strip()]
t = [f for f in figs if f["canonical_id"] == "2B-1" and f["kind"] == "Table"]
t.sort(key=lambda f: f["page_pdf"])
print(f"{len(t)} crops for Table 2B-1\n")
for f in t:
    print(f"pdf p{f['page_pdf']}  printed p{f['page_printed']}  "
          f"sheet={f.get('sheet')}/{f.get('sheet_of')}  {f['image_path'].split('/')[-1]}")
    display(IPImage(filename=f["image_path"], width=560))

### 8.9 Multi-sheet figures

In [ ]:
# CELL 34
import json, collections
figs = [json.loads(l) for l in open(CFG.figures_jsonl) if l.strip()]
canon = collections.defaultdict(list)
for f in figs:
    canon[(f["kind"], f["canonical_id"])].append(f)      # key on BOTH — ids collide across kinds

multi = {k: v for k, v in canon.items() if len(v) > 1}
print(f"canonical entities : {len(canon)}")
print(f"multi-sheet        : {len(multi)}")
print(f"sheets beyond first: {sum(len(v) - 1 for v in multi.values())}")
print(f"widest             : {sorted(((len(v), k[1]) for k, v in multi.items()), reverse=True)[:5]}")
print()
print(f"caps: sheets/figure={CFG.max_sheets_per_figure}  "
      f"images/request={CFG.max_images_total}  pages reserved={CFG.max_page_images}")
covered = sum(1 for v in canon.values() if len(v) <= CFG.max_sheets_per_figure)
print(f"entities fully shown under the per-figure cap: {covered}/{len(canon)}")

# ── allocation check ───────────────────────────────────────────────────────
# Selection is ROUND-ROBIN: sheet 1 of every figure, then sheet 2 of every
# figure, and so on. Greedy allocation let one 8-sheet table eat the budget and
# starve later figures, which cost more than it gained (TB009 went from full
# credit to abstaining). Every retrieved figure must keep at least sheet 1.
def simulate(sheet_counts, cap=None, per_fig=None, pages=1, reserve=None):
    cap     = cap or CFG.max_images_total
    per_fig = per_fig or CFG.max_sheets_per_figure
    reserve = CFG.max_page_images if reserve is None else reserve
    per = [list(range(min(n, per_fig))) for n in sheet_counts]
    budget = max(0, cap - min(pages, reserve))
    picked, depth = [], 0
    while len(picked) < budget and any(len(s) > depth for s in per):
        for fi, s in enumerate(per):
            if depth < len(s) and len(picked) < budget:
                picked.append((fi, depth))
        depth += 1
    shown = len({fi for fi, _ in picked})
    return len(picked), shown, len(sheet_counts)

for label, counts in [("TB009-like: 8-sheet table + 2 figures", [8, 6, 3]),
                      ("one 8-sheet table alone",               [8]),
                      ("five single-sheet figures",             [1, 1, 1, 1, 1])]:
    n, shown, total = simulate(counts)
    ok = "OK" if shown == total else "STARVED"
    print(f"  {label:<40} {n:>2} imgs, {shown}/{total} figures  {ok}")

## 9. PARKED — do not run

MUTCD-150 is retired as a VINE benchmark. The retrieval-size sweep is parked:
`retrieve()` stays at 6 and nothing in `mrag/vine/` calls it.

In [ ]:
# CELL 35
# Retrieval-size sweep. NO model generation: retrieval only, so this is cheap.
# Metrics are defined here in code, because the repo has no implementation of
# the Recall@5 / evidence-sufficiency numbers quoted in the write-ups.
import json, re, time, collections
from mrag.config import CFG

GOLD = "/content/Beyond_RAG_repo/evaluation/gold/mutcd_benchmark_gold_v1_1_msdi.jsonl"
gold = [json.loads(l) for l in open(GOLD) if l.strip()]

def expand_sections(s):
    """'2B.12-2B.17' -> ['2B.12' ... '2B.17']; plain ids pass through."""
    m = re.match(r"^(\d[A-Z])\.(\d{2})-(?:\d[A-Z]\.)?(\d{2})$", s)
    if not m:
        return [s]
    a, b = int(m.group(2)), int(m.group(3))
    return [f"{m.group(1)}.{i:02d}" for i in range(a, b + 1)]

def norm_fig(x):
    return re.sub(r"\s+", " ", str(x)).strip().replace("\u2011", "-").replace("\u2013", "-")

answerable = [q for q in gold if q.get("answerable")]
print(f"{len(gold)} questions, {len(answerable)} answerable\n")

KS = [4, 6, 8, 10, 12, 16, 20]
original_k = CFG.top_k_after_rerank
rows = []
try:
    for k in KS:
        CFG.top_k_after_rerank = k
        sec_rec, full_cov, prec, chars, fig_rec = [], [], [], [], []
        t0 = time.time()
        for q in answerable:
            r = pipeline.retriever.retrieve(q["question"])
            got_secs = [c.get("section_id", "") for c in r.chunks]
            want = {s for x in (q.get("sections") or []) for s in expand_sections(x)}
            if want:
                hit = want & set(got_secs)
                sec_rec.append(len(hit) / len(want))
                full_cov.append(1.0 if hit == want else 0.0)
                prec.append(sum(1 for s in got_secs if s in want) / max(1, len(got_secs)))
            chars.append(sum(min(len(c.get("text", "")), CFG.max_chunk_chars_in_prompt)
                             for c in r.chunks))
            want_fig = {norm_fig(x) for x in (q.get("figures") or []) + (q.get("tables") or [])}
            if want_fig:
                got_fig = {norm_fig(f.get("figure_id", "")) for f in r.figures}
                fig_rec.append(len(want_fig & got_fig) / len(want_fig))
        mean = lambda v: sum(v) / len(v) if v else float("nan")
        chars.sort()
        rows.append({
            "k": k,
            "section_recall": mean(sec_rec),
            "all_gold_sections": mean(full_cov),
            "context_precision": mean(prec),
            "figure_recall": mean(fig_rec),
            "median_prompt_chars": chars[len(chars) // 2],
            "secs_per_q": (time.time() - t0) / len(answerable),
        })
        print(f"k={k:<3} section recall {rows[-1]['section_recall']:.3f} | "
              f"all gold sections {rows[-1]['all_gold_sections']:.3f} | "
              f"precision {rows[-1]['context_precision']:.3f} | "
              f"figure recall {rows[-1]['figure_recall']:.3f} | "
              f"prompt chars {rows[-1]['median_prompt_chars']:>6} | "
              f"{rows[-1]['secs_per_q']:.2f}s/q")
finally:
    CFG.top_k_after_rerank = original_k
    print(f"\ntop_k_after_rerank restored to {CFG.top_k_after_rerank}")

out = CFG.base_dir / "retrieval_size_sweep.json"
out.write_text(json.dumps(rows, indent=2))
print("saved ->", out)


In [ ]:
# CELL 36
# The runner is a Python API, not a CLI.
import sys
REPO = "/content/Beyond_RAG_repo"
sys.path.insert(0, f"{REPO}/benchmarks/mutcd150/v1")

from mutcd_benchmark_runner import run_benchmark
from mrag.ask import ask
from mrag.config import CFG

BENCH = f"{REPO}/benchmarks/mutcd150/v1/mutcd_benchmark_questions_v1.jsonl"
OUT   = CFG.base_dir / "benchmark_runs"          # Drive/MyDrive/Beyond_RAG/benchmark_runs

paths = run_benchmark(
    CFG=CFG,
    ask_fn=ask,
    questions_path=BENCH,
    output_root=OUT,
    run_id="beyond_rag_002_roundrobin",          # NEW id — 001 used greedy allocation
    models=[{"alias": "fable", "selector": "frontier_claude", "provider": "anthropic"}],
    prompt_style="fewshot",

    # Smoke-test the regression first. Run these three, check TB009 answers
    # instead of abstaining, THEN comment this line out for the full 150.
    #question_ids=["TB008", "TB009", "TB026"],

    resume=True,
)
for k, v in paths.items():
    print(f"{k:<24} {v}")